###### ds_rank0与vllm_ranks之间的通讯
- PPO-Actor的ds_rank0需要和all_vllm_ranks进行通讯，传递最新的PPO-Actor权重，例如以下ds_rank0要把完整的权重broadcast给16个vllm_ranks：

###### 创建通信组

# step 1
- 代码来自：https://github.com/OpenRLHF/OpenRLHF/blob/bb46342711a203c457df2fbca5967fd0549557e0/openrlhf/trainer/ray/ppo_actor.py#L58
- <font color='red'>这段代码执行在PPO-Actor0（ds_rank0）所在的worker进程中。这个worker进程将通过handler引用，触发远端每个vllm_engine上的init_process_group操作，并将ds_rank0纳入通讯组</font>

In [ ]:
import math
import os
import socket
from typing import Callable, Dict, List

import deepspeed
import ray
import torch
from transformers.trainer import get_scheduler

from openrlhf.datasets import PromptDataset, SFTDataset
from openrlhf.models import Actor
from openrlhf.trainer import PPOTrainer
from openrlhf.trainer.ppo_utils import Experience, RemoteExperienceMaker
from openrlhf.utils import blending_datasets, get_tokenizer
from openrlhf.utils.deepspeed import DeepspeedStrategy
from openrlhf.utils.distributed_util import init_process_group

from .launcher import BasePPORole


class ActorPPOTrainer(PPOTrainer):
    def __init__(
        self,
        *args,
        vllm_engines: List = None,
        remote_rm_url: List[str] = None,
        critic_train_remote: bool = False,
        **kwargs,
    ):
        """PPOTrainer for ray.

        Args:
            vllm_engines (List, optional): vllm engines for text generation, if not specified, generate text by actor model directly. Defaults to None.
            critic_train_remote (bool, optional): whether this actor should triger corresponding critic model training. Defaults to False.
        """
        super().__init__(*args, **kwargs)
        self.remote_rm_url = remote_rm_url
        self.vllm_engines = vllm_engines
        self.critic_train_remote = critic_train_remote

        self.experience_maker = RemoteExperienceMaker(
            self.actor,
            self.critic,
            self.reward_model,
            self.initial_model,
            self.tokenizer,
            self.prompt_max_len,
            self.kl_ctl,
            self.strategy,
            self.remote_rm_url,
            self.reward_fn,
            vllm_engines=self.vllm_engines,
            packing_samples=self.strategy.args.packing_samples,
        )

        # Create torch group with deepspeed rank 0 and all vllm ranks
        # to update vllm engine's weights after each training stage.
        #
        # Say we have 3 vllm engines and eache of them has 4 GPUs,
        # then the torch group is:
        # [    0,      1, 2, 3, 4,  5, 6, 7, 8,  9, 10, 11, 12]
        # |ds rank 0 |  engine-0  |  engine-1  |   engine-2   |
        #
        # For ZeRO-1/2:
        #   1. Broadcast parameters from rank 0 to all vllm engines
        # For ZeRO-3:
        #   1. AllGather paramters to rank 0
        #   2. Broadcast parameters from rank 0 to all vllm engines
        
            # =====================================================================
            # 遍历每个vllm_engines，将其下的每个vllm_rank添加进通讯组中，这里又分成两步：
            # 1. engine.init_process_group.remote(...)：
            #    首先，触发远程vllm_engine的init_process_group方法
            #
            # 2. 远程vllm_engine是一个包装过的vllm实例，它的init_process_group
            #    方法将进一步触发这个vllm实例下的各个worker进程（见4.4图例），
            #    最终是在这些worker进程上执行“将每个vllm_rank"添加进ds_rank0通讯组的工作
            # =====================================================================
        if self.vllm_engines is not None and torch.distributed.get_rank() == 0:
            
            master_address = ray._private.services.get_node_ip_address()
            with socket.socket() as sock:
                sock.bind(("", 0))
                master_port = sock.getsockname()[1]

            vllm_num_engines, vllm_tensor_parallel_size = (
                self.strategy.args.vllm_num_engines,
                self.strategy.args.vllm_tensor_parallel_size,
            )
            world_size = vllm_num_engines * vllm_tensor_parallel_size + 1

            backend = getattr(self.strategy.args, "vllm_sync_backend", "nccl")
            # https://github.com/OpenRLHF/OpenRLHF/issues/313
            import vllm

            if not vllm.__version__ == "0.4.2" and not vllm.__version__ >= "0.6.4":
                backend = "gloo"
                print(
                    "Warning: using --vllm_sync_backend=gloo for `not vLLM version == 0.4.2 and not vllm.__version__ >= 0.6.4`"
                )

            refs = [
                engine.init_process_group.remote(
                    # ds_rank0所在node addr
                    master_address, 
                    # ds_rank0所在node port
                    master_port,
                    # 该vllm_engine的第一个rank在"ds_rank0 + all_vllm_ranks“中的global_rank，
                    # 该值将作为一个offset，以该值为起点，可以推算出该vllm_engine中其余vllm_rank的global_rank
                    i * vllm_tensor_parallel_size + 1, 
                    world_size,
                    "openrlhf",
                    backend=backend,
                )
                for i, engine in enumerate(self.vllm_engines)
            ]
            
            # =====================================================================
            # 将ds_rank0添加进通讯组中
            # =====================================================================
            self._model_update_group = init_process_group(
                backend=backend,
                init_method=f"tcp://{master_address}:{master_port}",
                world_size=world_size,
                rank=0,
                group_name="openrlhf",
            )

            # =====================================================================
            # 确保all_vllm_ranks都已添加进通讯组中
            # =====================================================================
            ray.get(refs)

        torch.distributed.barrier()

    def ppo_train(self, global_steps):
        # 1. ensure all experience makers done
        self.experience_maker.flush()
        torch.distributed.barrier()

        # 2. triger remote critic model training
        if self.critic_train_remote:
            critic_status_ref = self.critic.fit.remote()

        # 3. actor model training
        if global_steps > self.freezing_actor_steps:
            status = super().ppo_train(global_steps)

            # 4. broadcast weights to vllm engines
            if self.vllm_engines is not None:
                torch.distributed.barrier()
                self._broadcast_to_vllm()
        else:
            status = {}

        # 5. wait remote critic model training done
        if self.critic_train_remote:
            status.update(ray.get(critic_status_ref))
        torch.distributed.barrier()

        return status

    def training_step(self, experience: Experience, global_steps) -> Dict[str, float]:
        return self.training_step_actor(experience)

    def _broadcast_to_vllm(self):
        # avoid OOM
        torch.cuda.empty_cache()
        model = self.actor.model.module
        count, num_params = 0, len(list(model.named_parameters()))
        for name, param in model.named_parameters():
            count += 1  # empty_cache at last param

            # Fire all vllm engines for broadcast
            if torch.distributed.get_rank() == 0:
                shape = param.shape if self.strategy.args.zero_stage != 3 else param.ds_shape
                refs = [
                    engine.update_weight.remote(name, dtype=param.dtype, shape=shape, empty_cache=count == num_params)
                    for engine in self.vllm_engines
                ]

            # For ZeRO-3, allgather sharded parameter and broadcast to all vllm engines by rank 0
            with deepspeed.zero.GatheredParameters([param], enabled=self.strategy.args.zero_stage == 3):
                if torch.distributed.get_rank() == 0:
                    torch.distributed.broadcast(param.data, 0, group=self._model_update_group)
                    ray.get(refs)

    def _save_checkpoint(self, args, tag, client_states):
        # call remote critic
        if self.critic_train_remote:
            ref = self.critic.save_checkpoint.remote(tag)
        self.strategy.save_ckpt(
            self.actor.model,
            os.path.join(args.ckpt_path, "_actor"),
            tag,
            args.max_ckpt_num,
            args.max_ckpt_mem,
            client_states,
        )
        # wait
        if self.critic_train_remote:
            ray.get(ref)


@ray.remote(num_gpus=1)
class ActorModelRayActor(BasePPORole):
    def init_model_from_pretrained(self, strategy: DeepspeedStrategy, pretrain):
        args = strategy.args
        self._setup_distributed(strategy)

        actor = Actor(
            pretrain,
            use_flash_attention_2=strategy.args.flash_attn,
            bf16=strategy.args.bf16,
            load_in_4bit=strategy.args.load_in_4bit,
            lora_rank=strategy.args.lora_rank,
            lora_alpha=strategy.args.lora_alpha,
            target_modules=strategy.args.target_modules,
            lora_dropout=strategy.args.lora_dropout,
            ds_config=strategy.get_ds_train_config(is_actor=True),
            packing_samples=strategy.args.packing_samples,
        )
        strategy.print(actor)

        # configure tokenizer
        self.tokenizer = get_tokenizer(
            pretrain, actor.model, "left", strategy, use_fast=not strategy.args.disable_fast_tokenizer
        )

        if args.enable_ema:
            ema_model = Actor(
                pretrain,
                use_flash_attention_2=strategy.args.flash_attn,
                bf16=strategy.args.bf16,
                load_in_4bit=strategy.args.load_in_4bit,
                ds_config=strategy.get_ds_eval_config(offload=True),
                packing_samples=strategy.args.packing_samples,
            )
        else:
            ema_model = None

        # configure optimizer
        actor_optim = strategy.create_optimizer(
            actor, lr=args.actor_learning_rate, betas=strategy.args.adam_betas, weight_decay=args.l2
        )

        # prepare_datasets
        self.prepare_datasets()

        # configure scheduler
        self.num_update_steps_per_episodes = (
            len(self.prompts_dataset) * args.n_samples_per_prompt // args.train_batch_size * args.max_epochs
        )
        max_steps = math.ceil(args.num_episodes * self.num_update_steps_per_episodes)
        self._max_steps = max_steps

        actor_scheduler = get_scheduler(
            "cosine_with_min_lr",
            actor_optim,
            num_warmup_steps=math.ceil(max_steps * args.lr_warmup_ratio),
            num_training_steps=max_steps,
            scheduler_specific_kwargs={"min_lr": args.actor_learning_rate * 0.1},
        )

        if args.gradient_checkpointing:
            actor.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": args.gradient_checkpointing_use_reentrant}
            )

        # prepare models/optimizers...
        self.actor, self.actor_optim, self.actor_scheduler = strategy.prepare(
            (actor, actor_optim, actor_scheduler),
            is_rlhf=True,
        )

        if ema_model:
            ema_model._offload = True
            self.ema_model = strategy.prepare(ema_model, is_rlhf=True)
        else:
            self.ema_model = None

        # load checkpoint
        self.consumed_samples = 0
        ckpt_path = os.path.join(args.ckpt_path, "_actor")
        if args.load_checkpoint and os.path.exists(ckpt_path):
            _, states = strategy.load_ckpt(self.actor.model, ckpt_path)
            self.consumed_samples = states["consumed_samples"]
            strategy.print(f"Loaded the checkpoint: {ckpt_path}, consumed_samples: {self.consumed_samples}")

    def prepare_datasets(self):
        strategy = self.strategy
        args = self.strategy.args

        # prepare datasets
        prompts_data = blending_datasets(
            args.prompt_data,
            args.prompt_data_probs,
            strategy,
            args.seed,
            max_count=args.max_samples,
            return_eval=False,
            train_split=args.prompt_split,
        )
        prompts_data = prompts_data.select(range(min(args.max_samples, len(prompts_data))))
        self.prompts_dataset = PromptDataset(
            prompts_data, self.tokenizer, strategy, input_template=args.input_template
        )
        self.prompts_dataloader = strategy.setup_dataloader(
            self.prompts_dataset, args.rollout_batch_size // strategy.world_size, True, True
        )

        if args.pretrain_data:
            pretrain_data = blending_datasets(
                args.pretrain_data,
                args.pretrain_data_probs,
                strategy,
                args.seed,
                return_eval=False,
                train_split=args.pretrain_split,
            )
            pretrain_max_len = args.max_len if args.max_len else args.prompt_max_len + args.generate_max_len
            pretrain_dataset = SFTDataset(
                pretrain_data.select(
                    range(
                        min(
                            len(pretrain_data), args.max_epochs * len(self.prompts_dataset) * args.n_samples_per_prompt
                        )
                    )
                ),
                self.tokenizer,
                pretrain_max_len,
                strategy,
                pretrain_mode=True,
            )
            self.pretrain_dataloader = itertools.cycle(
                iter(
                    strategy.setup_dataloader(
                        pretrain_dataset,
                        args.micro_train_batch_size,
                        True,
                        True,
                        pretrain_dataset.collate_fn,
                    )
                )
            )
        else:
            self.pretrain_dataloader = None

    def max_steps(self):
        """Return the maximum number of steps."""
        return self._max_steps

    def fit(
        self,
        critic_model: ray.actor.ActorHandle,
        initial_model: ray.actor.ActorHandle,
        reward_model: List[ray.actor.ActorHandle],
        remote_rm_url: List[str] = None,
        reward_fn: Callable[[List[torch.Tensor]], torch.Tensor] = None,
        vllm_engines: List[ray.actor.ActorHandle] = None,
        critic_train_remote: bool = False,
    ):
        """Train actor model with prompt datasets."""
        strategy = self.strategy
        args = self.strategy.args

        # configure Trainer
        trainer = ActorPPOTrainer(
            strategy,
            self.actor,
            critic_model,
            reward_model,
            initial_model,
            ema_model=self.ema_model,
            actor_optim=None,
            critic_optim=None,
            actor_scheduler=self.actor_scheduler,
            critic_scheduler=None,
            remote_rm_url=remote_rm_url,
            reward_fn=reward_fn,
            vllm_engines=vllm_engines,
            max_epochs=args.max_epochs,
            micro_train_batch_size=args.micro_train_batch_size,
            micro_rollout_batch_size=args.micro_rollout_batch_size,
            gradient_checkpointing=args.gradient_checkpointing,
            critic_train_remote=critic_train_remote,
            tokenizer=self.tokenizer,
            prompt_max_len=args.prompt_max_len,
            value_clip=args.value_clip,
            eps_clip=args.eps_clip,
            gamma=args.gamma,
            lambd=args.lambd,
            init_kl_coef=args.init_kl_coef,
            kl_target=args.kl_target,
            ema_beta=0.992,
            ptx_coef=args.ptx_coef,
            max_norm=args.max_norm,
            # fro GPT generation
            do_sample=True,
            max_new_tokens=args.generate_max_len,
            max_length=args.max_len,
            temperature=args.temperature,
            top_p=args.top_p,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        # broadcast checkpoint
        ckpt_path = os.path.join(args.ckpt_path, "_actor")
        if args.load_checkpoint and os.path.exists(ckpt_path) and not vllm_engines is None:
            torch.distributed.barrier()
            trainer._broadcast_to_vllm()

        trainer.fit(
            args,
            self.prompts_dataloader,
            self.pretrain_dataloader,
            self.consumed_samples,
            self.num_update_steps_per_episodes,
        )

    def save_model(self):
        args = self.strategy.args

        # save model checkpoint after fitting on only rank0
        self.strategy.save_model(
            self.ema_model if args.enable_ema else self.actor,
            self.tokenizer,
            args.save_path,
        )

这段代码是 OpenRLHF 框架中实现高效权重同步的核心逻辑。<font color='red'>它的作用是构建一个跨越“训练”与“推理”两个异构系统的统一分布式通信组（Process Group）。</font>

简单来说，它把<font color='red'>负责训练的 DeepSpeed Rank 0 </font>和<font color='red'>负责推理的 所有 vLLM Worker </font>拉进了同一个聊天室，<font color='green'>以便直接将训练好的权重广播给推理引擎，而无需经过 CPU 内存中转。 </font>

以下是详细的代码逻辑与架构解析：

## 核心目标：构建统一通信组
<font color='red'>在 RLHF 流程中，Actor 模型（训练端）更新权重后，需要同步给 vLLM（推理端）以便进行下一轮采样。</font>
- <font color='red'>传统做法：</font>将权重从 GPU -> CPU -> 网络 -> CPU -> GPU，速度慢。
- <font color='red'>OpenRLHF 做法：</font><font color='green'>利用 PyTorch 分布式通信库（NCCL），直接在 GPU 之间进行数据传输。为此，必须让训练进程和推理进程在同一个“通讯组”里。</font>

### 全局 Rank 映射图解

代码注释中清晰地展示了这个虚拟通讯组的结构：

| 全局 Rank ID | 0 | 1, 2, 3, 4 | 5, 6, 7, 8 | 9, 10, 11, 12 |
| :--- | :---: | :---: | :---: | :---: |
| 角色 | DeepSpeed Rank 0 | vLLM Engine 0 | vLLM Engine 1 | vLLM Engine 2 |
| 功能 | <font color='red'>权重发送者 (Source) </font>| 权重接收者 (Target) | 权重接收者 (Target) | 权重接收者 (Target) |


## 代码逻辑分步解析

这段代码运行在 DeepSpeed Rank 0 进程上（<font color='red'>由 torch.distributed.get_rank() == 0 保证</font>）。

### 第一步：计算全局通讯组大小

In [ ]:
world_size = vllm_num_engines * vllm_tensor_parallel_size + 1

- vllm_num_engines * vllm_tensor_parallel_size：计算所有 vLLM 推理进程的总数。例如 3 个引擎，每个 TP=4，共 12 个推理进程。
- + 1：加上 DeepSpeed 的主进程（Rank 0）。
- 结果：整个通讯组共有 13 个成员（Rank 0 ~ 12）。
### 第二步：远程触发 vLLM 加入通讯组

In [ ]:
refs = [
    engine.init_process_group.remote(
        master_address, 
        master_port,
        i * vllm_tensor_parallel_size + 1,  # <--- 关键：Rank Offset
        world_size,
        "openrlhf",
        backend=backend,
    )
    for i, engine in enumerate(self.vllm_engines)
]

- <font color='red'>engine.init_process_group.remote(...)：这是一个 Ray 远程调用。DeepSpeed Rank 0 命令每一个 vLLM Engine（Ray Actor）去初始化它们的分布式环境。</font>
- <font color='red'>rank_offset (关键逻辑)：</font>
  - 代码：i * vllm_tensor_parallel_size + 1
  - 含义：vLLM 内部的 Worker 原本 Rank 是 0, 1, 2, 3。但在我们这个“统一大群”里，它们不能和 DeepSpeed Rank 0 冲突，也不能不同引擎间冲突。
  - 计算：
    - Engine 0 的 Workers 映射为全局 Rank 1, 2, 3, 4 (Offset = 1)
    - Engine 1 的 Workers 映射为全局 Rank 5, 6, 7, 8 (Offset = 5)
    - Engine 2 的 Workers 映射为全局 Rank 9, 10, 11, 12 (Offset = 9)
  - <font color='red'>这个 Offset 确保了每个 GPU 在整个通讯组中拥有唯一的“身份证号”。</font>

### 第三步：本地 DeepSpeed 进程加入通讯组

In [ ]:
self._model_update_group = init_process_group(
    backend=backend,
    init_method=f"tcp://{master_address}:{master_port}",
    world_size=world_size,
    rank=0,  # DeepSpeed Rank 0 在这个新组里依然是 Rank 0
    group_name="openrlhf",
)

- 这是标准的 PyTorch 分布式初始化。
- DeepSpeed Rank 0 作为主节点（Master），监听 master_address:master_port，等待 vLLM 的 Workers 连接上来。

### 第四步：同步等待所有进程就绪

In [ ]:
ray.get(refs)

- 作用：阻塞当前进程，直到所有远程的 vLLM Engine 都完成了 init_process_group 操作。
- 为什么重要：分布式通信是严格的“集合操作”。如果 Rank 0 试图广播数据，但 Rank 1-12 还没连接好，程序会直接死锁（Hang）。ray.get(refs) 确保了所有成员都“入座”后，才进行下一步。

## 为什么区分 ZeRO-1/2 和 ZeRO-3？

代码注释中提到了针对不同 ZeRO 策略的处理：
- ZeRO-1/2：
    - 模型参数主要保存在 Rank 0 上（或者 Rank 0 拥有完整副本）。
    - 操作：<font color='red'>Rank 0 直接 Broadcast（广播）参数给所有 vLLM Ranks。</font>
- ZeRO-3：
    - 模型参数被切分（Sharded）在所有 DeepSpeed 训练卡上，Rank 0 此时只有部分参数。
    - 操作：
        - <font color='red'>AllGather：Rank 0 先从其他训练卡（Rank 1, 2...）把完整的模型参数收集齐。</font>
        - <font color='red'>Broadcast：Rank 0 再把完整的参数广播给 vLLM。</font>
        
## 总结
<font color='red'>这段代码实现了“训练-推理”直通链路的建立：</font>

1. 统一编址：通过 rank_offset 解决了训练任务和推理任务 Global Rank 冲突的问题。
2. 跨框架通信：<font color='red'>利用 PyTorch 原生的 init_process_group，让 DeepSpeed（训练框架）和 vLLM（推理框架）在底层 NCCL 层面握手。</font>
3. 零拷贝同步：建立这个组之后，权重更新可以直接通过 GPU Direct 进行，极大提升了 RLHF 的训练吞吐量。

# step 2

- 代码来自：https://github.com/OpenRLHF/OpenRLHF/blob/bb46342711a203c457df2fbca5967fd0549557e0/openrlhf/trainer/ray/vllm_worker_wrap.py#L11
- <font color='red'>这段代码实际运行在每个vllm_engine（即每个包装后的vllm实例）下的worker进程内。</font>
- 例如tp_size=2，那么每个vllm实例下就有2个worker进程，这两个worker进程都会运行这段代码。

In [ ]:
import torch
from vllm.worker.worker import Worker

from openrlhf.utils.distributed_util import init_process_group
from openrlhf.utils.logging_utils import init_logger

logger = init_logger(__name__)


class WorkerWrap(Worker):
    def init_process_group(self, master_address, master_port, rank_offset, world_size, group_name, backend="nccl"):
       
        """Init torch process group for model weights update"""
        
        assert torch.distributed.is_initialized(), f"default torch process group must be initialized"
        assert group_name != "", f"group name must not be empty" # 必须设置所在的进程组
        # =====================================================================
        # torch.distributed.get_rank(): 在当前vllm_engine内部的rank，
        #                               例如在tp_size = 2时，这个值要么是0，要么是1
        # rank_offset：当前vllm_engine中的第一个rank在“ds_rank0 + all_vllm_ranks"中的global_rank
        # 两者相加：最终得到当前rank在“ds_rank0 + all_vllm_ranks"中的global_rank
        # =====================================================================
        rank = torch.distributed.get_rank() + rank_offset
        
        self._model_update_group = init_process_group(
            backend=backend,
            init_method=f"tcp://{master_address}:{master_port}",
            world_size=world_size,
            rank=rank,
            group_name=group_name,
        )
        ...

# init_process_group

这段代码定义了一个 init_process_group 方法，其核心目的是在一个已经初始化的分布式环境中，创建一个新的、独立的进程组（Process Group），专门用于模型权重的更新。

<font color='red'>这通常出现在复杂的分布式训练框架中，例如将 DeepSpeed 与 vLLM 引擎结合使用时。为了隔离不同组件（如优化器、模型并行）之间的通信，避免相互干扰，需要创建多个独立的进程组。</font>

下面我将分步详细解释这段代码的逻辑。

## 🎯 代码目的与背景
在大型分布式训练任务中，不同的操作可能需要不同的进程组。例如：
- 数据并行组 (Data Parallel Group): 用于在不同节点间同步梯度。
- 张量并行组 (Tensor Parallel Group): <font color='red'>用于在单个节点内的多个GPU间分割模型层。</font>
- 模型更新组 (Model Update Group): 如这段代码所示，专门用于同步更新后的模型权重。

这段代码就是为了创建最后一个“模型更新组”。

## 🔍 参数详解

这个方法接收了初始化一个新进程组所需的所有关键参数：

- <font color='red'>master_address (str):</font> 主节点（rank 0 所在的机器）的 IP 地址。<font color='red'>所有其他节点都会连接到这个地址来完成初始化握手。</font>
- <font color='red'>master_port (int):</font> 主节点上用于通信的端口号。
- <font color='red'>rank_offset (int):</font> 这是一个非常关键的参数，用于计算当前进程在新进程组中的全局排名（global rank）。
- <font color='red'>world_size (int):</font> 新进程组中包含的总进程数量。
- <font color='red'>group_name (str):</font> 新进程组的名称，用于标识。
- <font color='red'>backend (str):</font> 通信后端，默认为 "nccl"。<font color='red'>NCCL 是 NVIDIA 提供的用于 GPU 间高效通信的库，是 GPU 分布式训练的首选。</font>

## ⚙️ 逻辑分步解析
### 前置检查 (Assertions)
- <font color='red'>assert torch.distributed.is_initialized():</font> <font color='green'>这行代码确保在执行此方法前，PyTorch 的默认分布式环境已经被初始化。这意味着当前的脚本已经在一个大的分布式集群中运行了。</font> 
- assert group_name != "": 确保为新进程组指定了一个有效的名称。
### 计算全局排名 (Global Rank Calculation)
这是整个方法最核心、最巧妙的部分。

In [ ]:
rank = torch.distributed.get_rank() + rank_offset

- <font color='red'>torch.distributed.get_rank()</font>: 获取当前进程在当前所在进程组（很可能是 vLLM 引擎内部的张量并行组）中的局部排名（local rank）。
    - 例如，如果 vLLM 的 tp_size=2，那么这个值在当前引擎内要么是 0，要么是 1。
- rank_offset: 这是一个偏移量，它代表了当前 vLLM 引擎在整个全局集群中的起始排名。
- 两者相加: <font color='red'>通过相加，将当前进程的局部排名转换为了它在整个全局集群中的全局排名。</font>

###### 举个例子：
假设你有 2 个节点，每个节点 1 个 GPU。

- DeepSpeed 进程 (rank 0): 负责优化器步骤。
- vLLM 引擎: 启动 2 个进程用于推理，它们的全局排名是 1 和 2。
    - 在 vLLM 引擎内部，这两个进程的局部排名是 0 和 1。
    - 对于 vLLM 引擎来说，它的 rank_offset 就是 1。
    - 因此，vLLM 内部 rank 为 0 的进程，其全局 rank 是 0 + 1 = 1。
    - vLLM 内部 rank 为 1 的进程，其全局 rank 是 1 + 1 = 2。
这样，新创建的进程组就能准确地包含全局排名为 1 和 2 的这两个进程。

### 初始化新的进程组

In [ ]:
from openrlhf.utils.distributed_util import init_process_group

self._model_update_group = init_process_group(
    backend=backend,
    init_method=f"tcp://{master_address}:{master_port}",
    world_size=world_size,
    rank=rank,
    group_name=group_name,
)

<font color='red'>这行代码调用了 torch.distributed 的初始化函数（代码中简写为 init_process_group，实际应为 torch.distributed.init_process_group），并传入了计算好的全局 rank 和其他参数。</font>

- init_method=f"tcp://...": 指定使用 TCP 方式进行初始化，所有进程通过指定的 master_address 和 master_port 进行连接和同步。
- 最终，一个名为 self._model_update_group 的新进程组被创建并保存下来。
    - <font color='red'>后续的集合通信操作（如 all_reduce, broadcast）就可以指定在这个组内进行，从而实现模型权重的独立更新，而不会影响到其他并行的通信任务。</font>
    
######  openrlhf.utils.distributed_util文件内容如下：

In [ ]:
from datetime import timedelta
from typing import Any, Optional, Union

import torch
import torch.distributed
from torch.distributed.distributed_c10d import (
    Backend,
    PrefixStore,
    Store,
    _new_process_group_helper,
    _world,
    default_pg_timeout,
    rendezvous,
)


# Copy from pytorch to allow creating multiple main groups.
# https://github.com/pytorch/pytorch/blob/main/torch/distributed/distributed_c10d.py
def init_process_group(
    backend: Union[str, Backend] = None,
    init_method: Optional[str] = None,
    timeout: Optional[timedelta] = None,
    world_size: int = -1,
    rank: int = -1,
    store: Optional[Store] = None,
    group_name: str = None,
    pg_options: Optional[Any] = None,
):
    assert (store is None) or (init_method is None), "Cannot specify both init_method and store."

    if store is not None:
        assert world_size > 0, "world_size must be positive if using store"
        assert rank >= 0, "rank must be non-negative if using store"
    elif init_method is None:
        init_method = "env://"

    if backend:
        backend = Backend(backend)
    else:
        backend = Backend("undefined")

    if timeout is None:
        timeout = default_pg_timeout

    # backward compatible API
    if store is None:
        rendezvous_iterator = rendezvous(init_method, rank, world_size, timeout=timeout)
        store, rank, world_size = next(rendezvous_iterator)
        store.set_timeout(timeout)

        # Use a PrefixStore to avoid accidental overrides of keys used by
        # different systems (e.g. RPC) in case the store is multi-tenant.
        store = PrefixStore(group_name, store)

    # NOTE: The pg_options parameter was renamed into backend_options in PyTorch 2.6.0
    # https://github.com/pytorch/pytorch/commit/a0c7029a75628cd5fa8df83c0de0ea98ee7fd844
    # We need to determine the appropriate parameter name based on PyTorch version
    pg_options_param_name = "backend_options" if str(torch.__version__) >= "2.6" else "pg_options"
    
    pg, _ = _new_process_group_helper(
        world_size,
        rank,
        [],
        backend,
        store,
        group_name=group_name,
        **{pg_options_param_name: pg_options},
        timeout=timeout,
    )

    _world.pg_group_ranks[pg] = {i: i for i in range(world_size)}

    return pg

### _broadcast_to_vllm（广播）
- <font color='red'>构建好通讯组，我们就可以从ds_rank0<b>广播PPO-Actor权重</b>到all_vllm_ranks上了</font>，这里也分成两步。

# Step1：PPO-Actor ds_rank0发送权重

- 代码在：https://github.com/OpenRLHF/OpenRLHF/blob/bb46342711a203c457df2fbca5967fd0549557e0/openrlhf/trainer/ray/ppo_actor.py#L146
- 这段代码运行在ds_rank0对应的worker进程中

In [ ]:
def _broadcast_to_vllm(self):
        # avoid OOM
        torch.cuda.empty_cache()
        model = self.actor.model.module
        count, num_params = 0, len(list(model.named_parameters()))
        
        for name, param in model.named_parameters():
            count += 1  # empty_cache at last param

            # Fire all vllm engines for broadcast
            if torch.distributed.get_rank() == 0:
                shape = param.shape if self.strategy.args.zero_stage != 3 else param.ds_shape
                refs = [
                    # 远端vllm_engine的每个rank上，初始化一个尺寸为shape的empty weight张量，
                    # 用于接收广播而来的权重
                    engine.update_weight.remote(name, dtype=param.dtype, shape=shape, empty_cache=count == num_params)
                    for engine in self.vllm_engines
                ]

            # For ZeRO-3, allgather sharded parameter and broadcast to all vllm engines by rank 0
            # ds_rank0发出权重（视是否使用zero3决定在发出前是否要做all-gather）
            with deepspeed.zero.GatheredParameters([param], enabled=self.strategy.args.zero_stage == 3):
                if torch.distributed.get_rank() == 0:
                    torch.distributed.broadcast(param.data, 0, group=self._model_update_group)
                    ray.get(refs) # 确保所有vllm_ranks接收权重完毕

###### _broadcast_to_vllm
- 这段代码定义了一个名为 _broadcast_to_vllm 的方法，
- <font color='red'>其核心功能是在一个复杂的分布式训练环境中（例如使用 DeepSpeed 进行训练，并用 vLLM 进行推理），将训练好的模型权重从 DeepSpeed 进程高效、安全地同步到 vLLM 引擎中。</font>

- <font color='red'>这个过程是为了确保 vLLM 引擎能够使用最新的模型参数进行后续的推理或采样</font>，这在 PPO（近端策略优化）等强化学习算法中非常常见。

###### 下面我将分步详细解释这段代码的逻辑。

## 🎯 代码目的与背景

在结合了 DeepSpeed 和 vLLM 的架构中，通常存在两个独立的进程组：

1. <font color='red'>DeepSpeed 进程组</font>: <font color='green'>负责模型的训练和权重更新</font>。为了节省显存，它可能会使用 ZeRO-3 等优化技术，将模型参数分片存储在不同的 GPU 上。
2. <font color='red'>vLLM 进程组</font>: <font color='green'>负责高效的模型推理</font>。它需要一个完整、统一的模型副本。

<font color='red'>_broadcast_to_vllm 方法的作用就是在这两个进程组之间搭建一座桥梁，将 DeepSpeed 侧更新后的权重“广播”给 vLLM 侧的所有进程。</font>

## ⚙️ 逻辑分步解析
### 1. 内存清理与准备

In [ ]:
# avoid OOM
torch.cuda.empty_cache()
model = self.actor.model.module
count, num_params = 0, len(list(model.named_parameters()))

- <font color='red'>_torch.cuda.empty_cache()</font>: 在开始传输大量数据前，首先清理 PyTorch 的 GPU 缓存，释放不必要的显存，以防止在后续操作中因显存不足（OOM, Out Of Memory）而失败。

- model = self.actor.model.module: <font color='red'>获取 DeepSpeed 包装后的实际模型对象。</font>

- count, num_params: <font color='red'>初始化一个计数器，用于追踪当前正在处理第几个参数，这对于在最后一个参数处理完后进行清理非常有用。</font>

### 2. <font color='red'>触发 vLLM 接收准备 (仅由 DeepSpeed 主进程执行)</font>

In [ ]:
# Fire all vllm engines for broadcast
if torch.distributed.get_rank() == 0:
    
    shape = param.shape if self.strategy.args.zero_stage != 3 else param.ds_shape
    refs = [
        engine.update_weight.remote(name, dtype=param.dtype, shape=shape, empty_cache=count == num_params)
        for engine in self.vllm_engines
    ]

- if torch.distributed.get_rank() == 0: <font color='red'>这个广播操作由 DeepSpeed 进程组的主进程（rank 0）来协调发起。</font>

- 确定参数形状: <font color='red'>代码需要判断是否启用了 ZeRO-3。</font>
    - 如果未启用 ZeRO-3，参数是完整的，直接使用 param.shape。
    - <font color='red'>如果启用了 ZeRO-3，参数是分片的，需要使用 param.ds_shape 来获取完整的参数形状，以便 vLLM 侧能创建正确大小的张量来接收。</font>
- 远程调用 vLLM: 通过 Ray 框架，异步调用所有 vLLM 引擎的 update_weight 方法。
    - engine.update_weight.remote(...): 这是一个非阻塞的远程调用。<font color='red'>它会告诉每个 vLLM 引擎：“准备好一个名为 name，数据类型为 dtype，形状为 shape 的空张量，准备接收数据。”</font>
    - refs: 这是一个包含所有远程调用返回的 ObjectRef 的列表，用于后续同步等待。
    
### 3. 处理 ZeRO-3 分片并广播权重

In [ ]:
# For ZeRO-3, allgather sharded parameter and broadcast to all vllm engines by rank 0
with deepspeed.zero.GatheredParameters([param], enabled=self.strategy.args.zero_stage == 3):
    if torch.distributed.get_rank() == 0: # 在主进程中
        
        torch.distributed.broadcast(param.data, 0, group=self._model_update_group)
        ray.get(refs)

这是整个方法最核心的部分，它巧妙地处理了 ZeRO-3 的复杂性。

- <font color='red'>with deepspeed.zero.GatheredParameters(...):</font>
    - <font color='red'>这是一个上下文管理器。如果启用了 ZeRO-3 (enabled=True)，它会自动将所有分片在其他 GPU 上的参数收集（all-gather）到当前 GPU 上，形成一个完整的参数副本。</font>
    - <font color='red'>如果未启用 ZeRO-3，这个上下文管理器不执行任何操作，param 本身就是完整的。</font>
    - 通过这种方式，代码在 with 块内部总是能访问到一个完整的 param。
- torch.distributed.broadcast(...):
    - <font color='red'>在 with 块内部，DeepSpeed 的主进程（rank 0）调用 PyTorch 的集合通信函数 broadcast。</font>
        - 它将完整的 param.data 从自己（src=0）广播出去。
    - group=self._model_update_group: <font color='red'>关键点在于，这个广播操作发生在一个专门的“模型更新进程组”内，这个组包含了 DeepSpeed 的主进程和所有 vLLM 的进程。</font>
- ray.get(refs):
    - <font color='red'>这是一个阻塞调用</font>。<font color='green'>DeepSpeed 的主进程会在这里等待，直到所有 vLLM 引擎都完成了它们的 update_weight 任务（即接收完权重并加载到模型中）。</font>
    - 这确保了在权重更新完成前，整个流程不会继续向下执行，保证了同步的正确性。


## 📡 vLLM 侧的对应操作

<font color='red'>在 vLLM 的每个工作进程中，都会执行一个与 update_weight 类似的函数来接收权重。</font>

###### 其核心逻辑如下：
1. 创建接收张量: 根据 DeepSpeed 主进程发来的 shape 和 dtype，在本地创建一个空的 CUDA 张量。
2. <font color='red'>接收广播</font>: <font color='green'>调用 torch.distributed.broadcast(weight, 0, group=self._model_update_group)。这个调用会与 DeepSpeed 主进程的广播操作配对，将数据接收过来。</font>
3. <font color='red'>更新模型</font>: 使用接收到的 weight 数据，更新 vLLM 内部模型的对应参数。

<b>通过这一套流程，模型权重就从使用 ZeRO-3 优化的 DeepSpeed 训练环境，完整、正确地同步到了用于推理的 vLLM 环境中。</b>

# Step2: 各个vllm_ranks接收权重

- 代码在：https://github.com/OpenRLHF/OpenRLHF/blob/bb46342711a203c457df2fbca5967fd0549557e0/openrlhf/trainer/ray/vllm_worker_wrap.py#L29
- <font color='red'>代码运行在每个vllm_engine(即每个包装后的vllm实例)下的各个worker进程中。例如tp_size = 2，那么每个vllm实例下有2个worker进程，这2个worker进程都会运行这段代码。</font>

In [ ]:
 def update_weight(self, name, dtype, shape, empty_cache=False):
        
        """Broadcast weight to all vllm workers from source rank 0 (actor model)"""
        if torch.distributed.get_rank() == 0:
            print(f"update weight: {name}, dtype: {dtype}, shape: {shape}")

        assert dtype == self.model_config.dtype, f"mismatch dtype: src {dtype}, dst {self.model_config.dtype}"
        # 创建同尺寸空张量用于接收ds_rank0广播来的权重
        weight = torch.empty(shape, dtype=dtype, device="cuda")
        
        '''
        接收权重
        '''
        torch.distributed.broadcast(weight, 0, group=self._model_update_group)
        # 使用接收到的权重进行更新
        self.model_runner.model.load_weights(weights=[(name, weight)])

        del weight

###### update_weight

这段代码定义了一个 update_weight 方法，<font color='red'>它是 vLLM 引擎侧用于接收并加载新模型权重的核心函数。</font>

- <font color='red'>这个方法是与您之前提供的 DeepSpeed 侧的 _broadcast_to_vllm 方法协同工作的“接收端”。</font>
- 当 DeepSpeed 的主进程发起权重广播时，vLLM 的所有工作进程都会执行这个方法来完成接收和更新。

###### 下面我将分步详细解释这段代码的逻辑。

## 🎯 代码目的与背景

在分布式强化学习（如 PPO）的架构中，模型训练（Actor）和模型推理（vLLM）通常由不同的进程组负责。
- <font color='red'>发送方 (DeepSpeed): 负责训练并更新模型权重。</font>
-  <font color='green'>接收方 (vLLM): 需要获取最新的权重，以便用最新的策略进行推理和生成数据。</font>

<b>update_weight 方法就是 vLLM 进程中的“接收器”，它监听来自 DeepSpeed 主进程的广播，并将接收到的权重加载到自己的模型中。</b>

## ⚙️ 逻辑分步解析
### 1. 接收参数与调试

In [ ]:
def update_weight(self, name, dtype, shape, empty_cache=False):
    """Broadcast weight to all vllm workers from source rank 0 (actor model)"""
    if torch.distributed.get_rank() == 0:
        print(f"update weight: {name}, dtype: {dtype}, shape: {shape}")

- 方法签名: 接收来自 DeepSpeed 主进程的指令，包括要更新的参数名 (name)、数据类型 (dtype)、形状 (shape) 以及一个可选的内存清理标志 (empty_cache)。
- 调试打印: 只有 vLLM 进程组的主进程（rank 0）会打印日志。这是一种常见的调试手段，避免所有进程都输出相同的信息，造成日志混乱。

### 2. 数据类型安全检查

In [ ]:
assert dtype == self.model_config.dtype, f"mismatch dtype: src {dtype}, dst {self.model_config.dtype}"

<b>这是一个至关重要的安全检查。</b>

- dtype: <font color='red'>发送方（DeepSpeed）传来的权重数据类型。</font>
- self.model_config.dtype: <font color='red'>vLLM 模型自身配置的数据类型（例如 torch.float16 或 torch.bfloat16）</font>。

- 目的: <font color='red'>确保发送方和接收方的数据类型完全一致</font>。如果类型不匹配（例如，发送 float32，接收方期望 float16），直接加载会导致计算错误或程序崩溃。assert 语句会在不匹配时立即中断程序并抛出清晰的错误信息。

### 3. 创建接收张量

In [ ]:
# 创建同尺寸空张量用于接收ds_rank0广播来的权重
weight = torch.empty(shape, dtype=dtype, device="cuda")

- torch.empty(...): 在 GPU 上创建一个指定 shape 和 dtype 的空张量。<font color='red'>这个张量是未初始化的，分配内存非常快。</font>
- 目的: <font color='red'>这个 weight 张量就像一个“空桶”，专门用来接收从 DeepSpeed 主进程“倒过来”的权重数据。</font>

### 4. 接收广播数据

In [ ]:
# 接收权重
torch.distributed.broadcast(weight, 0, group=self._model_update_group)

<font color='red'>这是与 DeepSpeed 侧通信的核心。</font>

- <font color='red'>torch.distributed.broadcast(...): 这是一个集合通信操作。</font>
- 发送方: <font color='red'>在 _broadcast_to_vllm 方法中，DeepSpeed 的主进程（rank 0）作为发送方。</font>
- 接收方: <font color='red'>在这里，vLLM 的所有进程（包括 rank 0 自己）都会调用这个函数。它们会等待并接收从源 rank 0 发来的数据。</font>
- group=self._model_update_group: <font color='red'>通信被限制在之前专门创建的 _model_update_group 进程组内。</font>这确保了广播操作不会干扰到其他并行的通信任务（如数据并行梯度同步）。

### 5. 加载权重到模型

In [ ]:
# 使用接收到的权重进行更新
self.model_runner.model.load_weights(weights=[(name, weight)])

- <font color='red'>self.model_runner.model: 这是 vLLM 内部实际用于推理的模型对象。</font>
- load_weights(...): 调用 vLLM 模型提供的接口，将刚刚接收到的 weight 张量，根据其 name，赋值给模型中对应的参数。
- weights=[(name, weight)]: <font color='red'>以列表形式传入，允许一次性更新多个权重，但在这里是逐个更新。</font>

### 6. 清理临时张量

In [ ]:
del weight

- del weight: 删除临时的 weight 张量。
- 目的: 权重数据现在已经成功地加载到了 self.model_runner.model 的内部参数中。这个临时的 weight 张量已经完成了它的使命，删除它可以立即释放其占用的 GPU 显存，这是一种良好的显存管理习惯。

